In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import missingno as msno
import seaborn as sns
from imblearn.over_sampling import SMOTE
from ucimlrepo import fetch_ucirepo
from io import BytesIO
import os


In [2]:
def uci_data_request(train_url,labels_url):
  response = requests.get(train_url)#training data
  data = pd.read_csv(BytesIO(response.content), sep=" ", header=None)
  response = requests.get(labels_url)#labels data
  labels_data = pd.read_csv(BytesIO(response.content), sep=" ", header=None)
  df = pd.merge(data, labels_data,left_index=True, right_index=True,how='inner')
  df.rename(columns={'0_x':'0','1_x':'1','0_y':'Pass/Fail','1_y':'Time'}, inplace = True)
  return df

train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
labels_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"

df = uci_data_request(train_url,labels_url)
df

,0,1,2,3,4,5,6,7,8,9,...,582,583,584,585,586,587,588,589,Pass/Fail,Time
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1,19/07/2008 11:55:00
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1,19/07/2008 12:32:00
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,19/07/2008 13:17:00
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1,19/07/2008 14:43:00
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1,19/07/2008 15:22:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,-0.0045,...,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,-1,16/10/2008 15:13:00
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,-0.0061,...,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,-1,16/10/2008 20:49:00
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,NaN,NaN,...,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,-1,17/10/2008 05:26:00
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,-0.0072,...,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,-1,17/10/2008 06:01:00


轉成 Minutely Data

In [ ]:
df2=df.copy()
df2['Time']=pd.to_datetime(df2['Time'],format='%d/%m/%Y %H:%M:%S',dayfirst=True,errors='coerce')
df2=df2.dropna(subset=['Time']).sort_values('Time')
cands=[c for c in df2.columns
       if str(c).lower().replace('/','').replace('_','') in {'passfail','passfailtime','passfailcol'}]
if 'Pass/Fail' in df2.columns: cands=['Pass/Fail']
elif 'PassFail' in df2.columns: cands=['PassFail']
pf_col=cands[0] if len(cands)>0 else None
num_cols=df2.select_dtypes(include=[np.number]).columns.tolist()
if pf_col in num_cols:
    num_cols.remove(pf_col)
def _mode_or_first(s):
    m=s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]
agg={c:'mean' for c in num_cols}
if pf_col: agg[pf_col]=_mode_or_first
g=df2.groupby('Time',as_index=True).agg(agg)

#每分鐘重採樣 + 線性插值（依時間），首尾也補
out=g.resample('T').mean()
if num_cols:
    out[num_cols]=out[num_cols].interpolate(method='time',limit_direction='both')
if pf_col:
    pf=g[pf_col].resample('T').first().ffill().bfill()
    # 若有非 {-1,1} 的值，規整成符號；再用鄰近值補完
    pf=np.sign(pf.where(~pd.isna(pf),np.nan))
    pf=pf.replace(0,np.nan).ffill().bfill().astype(int)
    out[pf_col]=pf
df_minutely = out.reset_index()
df_minutely

C:\Users\Sandy\AppData\Local\Temp\ipykernel_1964\3754116996.py:22: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  out=g.resample('T').mean()
C:\Users\Sandy\AppData\Local\Temp\ipykernel_1964\3754116996.py:26: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  pf=g[pf_col].resample('T').first().ffill().bfill()


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.930000,2564.000000,2187.733300,1411.126500,1.360200,100.0,97.613300,0.124200,1.500500,...,208.204500,0.500500,0.011800,0.003500,2.363000,0.009600,0.020100,0.0060,208.204500,-1
1,2008-07-19 11:56:00,3032.682703,2561.328108,2188.887054,1412.546341,1.345854,100.0,97.741138,0.124214,1.500395,...,208.204500,0.500538,0.012084,0.003554,2.419262,0.009600,0.020100,0.0060,208.204500,-1
2,2008-07-19 11:57:00,3034.435405,2558.656216,2190.040808,1413.966181,1.331508,100.0,97.868976,0.124227,1.500289,...,208.204500,0.500576,0.012368,0.003608,2.475524,0.009600,0.020100,0.0060,208.204500,-1
3,2008-07-19 11:58:00,3036.188108,2555.984324,2191.194562,1415.386022,1.317162,100.0,97.996814,0.124241,1.500184,...,208.204500,0.500614,0.012651,0.003662,2.531786,0.009600,0.020100,0.0060,208.204500,-1
4,2008-07-19 11:59:00,3037.940811,2553.312432,2192.348316,1416.805862,1.302816,100.0,98.124651,0.124254,1.500078,...,208.204500,0.500651,0.012935,0.003716,2.588049,0.009600,0.020100,0.0060,208.204500,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129248,2008-10-17 06:03:00,2911.586667,2504.926667,2183.170333,1760.545533,1.581000,100.0,94.232233,0.122033,1.462200,...,108.257533,0.499833,0.017900,0.003867,3.586633,0.021367,0.021733,0.0065,108.257533,-1
129249,2008-10-17 06:04:00,2919.920000,2491.385000,2186.238850,2048.953950,1.585200,100.0,91.949450,0.122400,1.462200,...,115.639250,0.499550,0.017950,0.003900,3.596850,0.018950,0.020350,0.0060,115.639250,-1
129250,2008-10-17 06:05:00,2928.253333,2477.843333,2189.307367,2337.362367,1.589400,100.0,89.666667,0.122767,1.462200,...,123.020967,0.499267,0.018000,0.003933,3.607067,0.016533,0.018967,0.0055,123.020967,-1
129251,2008-10-17 06:06:00,2936.586667,2464.301667,2192.375883,2625.770783,1.593600,100.0,87.383883,0.123133,1.462200,...,130.402683,0.498983,0.018050,0.003967,3.617283,0.014117,0.017583,0.0050,130.402683,-1


In [ ]:
rename_map={}
for c in df_minutely.columns:
    if c=='Time':
        rename_map[c]='time'
    elif c=='Pass/Fail':
        rename_map[c]='label'
    elif str(c).isdigit():
        rename_map[c]='v'+str(c)  # 0 -> v0, 1 -> v1, ..., 589 -> v589
df_minutely = df_minutely.rename(columns=rename_map)


In [ ]:
#output
df_minutely.to_csv(r"D:\Sandy\UCI-SECOM\df_minutely.csv",index=False)

In [5]:
df_list=np.array_split(df_minutely,15)
for i,df in enumerate(df_list,1):
    globals()[f"df_min{i}"]=df.copy()

c:\Users\Sandy\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [8]:
#output :df_min(拆分成15個檔案)
output_dir=r"D:\Sandy\UCI-SECOM\df_min_split"
os.makedirs(output_dir,exist_ok=True)
for i,df in enumerate(df_list,1):
    df.to_csv(fr"{output_dir}\df_min{i}.csv",index=False)
